# SFT stage 1 on a free GPU (Kaggle, or Colab free)

Trains the text-to-graph adapter (Qwen3.5-4B, LoRA) on a 16 GB T4 or P100, which has no bf16.
The recipe is the repository's, with one change: the base model is loaded 4-bit and the compute is fp16 (QLoRA).
Read `docs/sft-training.md`, section "Run it on Kaggle (free tier)", first.

What you need before running:

- **Kaggle**: a private Dataset with `vlm-floorplan-source.tar.gz` and `train.jsonl` (build it with `scripts/sft_pack_inputs.py`), added to this notebook as an input. Turn on the GPU accelerator and Internet.
- **Colab**: a Google Drive folder `vlm-floorplan-sft` with the same two files. Choose a GPU runtime. Runs are saved in the folder `runs` inside it.

The cells run top to bottom. Only the first one is meant to be edited. Start with `MODE = "smoke"`: it takes minutes, prints speed, memory and whether the loss stays finite in fp16, and trains nothing you keep.

In [ ]:
# ---- Settings: the only cell you edit -------------------------------------------------------
import time

T0 = time.time()
PLATFORM = "auto"  # "auto", "kaggle" or "colab"
MODE = "smoke"  # "smoke": SMOKE_STEPS optimizer steps, then stop. "train": the full run (resumes on its own).
RUN_NAME = "sft-stage1"
SESSION_HOURS = 12.0  # Kaggle: 12. Colab free: what you actually get, often 4 to 5.
SAVE_MARGIN_HOURS = 0.5  # training stops this long before the session ends, to save a snapshot
DRIVE_FOLDER = "vlm-floorplan-sft"  # Colab only: the Google Drive folder with the two input files
SMOKE_STEPS = 8  # fp16 skips its first few steps while the loss scale settles, so 4 is too few
BUILD_CAUSAL_CONV1D = True  # compiled from source: about 25 min per session on Kaggle
EXTRA_SETS = []  # more settings for this run, e.g. ["data.max_length=8192", "train.num_epochs=2"]

In [ ]:
# ---- Where things are -----------------------------------------------------------------------
import contextlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

if PLATFORM == "auto":
    PLATFORM = "kaggle" if Path("/kaggle/working").is_dir() else "colab"
if PLATFORM == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    INPUT_ROOTS, OUT, TMP = [DRIVE], DRIVE / "runs", Path("/content/vlmfp")
elif PLATFORM == "kaggle":
    # Everything in /kaggle/working is saved as the notebook output; /tmp is not.
    INPUT_ROOTS, OUT, TMP = [Path("/kaggle/input")], Path("/kaggle/working"), Path("/tmp/vlmfp")
else:
    raise ValueError(f"PLATFORM must be auto, kaggle or colab, not {PLATFORM!r}")
OUT.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUT / RUN_NAME
LOG_DIR = OUT / "logs"
LOG_DIR.mkdir(exist_ok=True)

SRC = TMP / "source"
# The source arrives in the input dataset because the repository is private. Colab keeps the
# tarball as it is; Kaggle unpacks an uploaded tarball into a folder of the same name.
if not SRC.exists():
    roots = [root for root in INPUT_ROOTS if root.is_dir()]
    tarballs = [p for root in roots for p in root.rglob("vlm-floorplan-source.tar.gz")]
    folders = [
        p
        for root in roots
        for p in root.rglob("vlm-floorplan-source")
        if (p / "pyproject.toml").is_file()
    ]
    if len(tarballs) + len(folders) != 1:
        raise FileNotFoundError(
            f"expected one vlm-floorplan-source under {INPUT_ROOTS}, found {tarballs + folders}"
        )
    if tarballs:
        with tarfile.open(tarballs[0]) as tar:
            tar.extractall(SRC, **({"filter": "data"} if hasattr(tarfile, "data_filter") else {}))
    else:
        shutil.copytree(folders[0], SRC)
sys.path.insert(0, str(SRC / "scripts"))
import sft_platform  # noqa: E402  (standard library only; from the unpacked source)

TRAIN_FILE = sft_platform.find_file(INPUT_ROOTS, "train.jsonl")
print(f"platform: {PLATFORM}\nsource:   {SRC}\ntraining: {TRAIN_FILE}\noutput:   {OUT}")

In [ ]:
# ---- GPU check: pick the precision, or refuse ------------------------------------------------
query = "name,memory.total,compute_cap,driver_version"
smi = subprocess.run(
    ["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader,nounits"],
    capture_output=True,
    text=True,
)
if smi.returncode != 0:
    raise RuntimeError(
        "No GPU found. Kaggle: Session options > Accelerator > GPU T4 x2. Colab: Runtime > Change runtime type > T4."
    )
gpus = [[field.strip() for field in line.split(",")] for line in smi.stdout.strip().splitlines()]
name, memory_mib, compute_cap, driver = gpus[0]
try:
    PLAN = sft_platform.plan_for_gpu(
        name, int(float(memory_mib)), tuple(int(n) for n in compute_cap.split("."))
    )
except sft_platform.UnsupportedGpuError as err:
    raise RuntimeError(f"STOP: {err}") from None
print(f"{len(gpus)} GPU(s), driver {driver}.")
print("Plan:", PLAN.reason)
if len(gpus) > 1:
    print(
        "Training uses GPU 0 only: the trainer would otherwise split every batch over both cards,"
    )
    print("which 4-bit loading does not support, and a batch of 1 gains nothing from it.")

In [ ]:
# ---- Install the ml dependency group (about 5 to 10 minutes) ---------------------------------
def run(cmd, *, log=None, **kwargs):
    """Run a command, show its output live, and also append it to a log file."""
    print("$", " ".join(map(str, cmd)), flush=True)
    with open(log, "a") if log else contextlib.nullcontext() as file:
        proc = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, **kwargs
        )
        for line in proc.stdout:
            print(line, end="", flush=True)
            if file:
                file.write(line)
    return proc.wait()


ENV_DIR = TMP / "venv"
ENV = {k: v for k, v in os.environ.items() if k not in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV")}
ENV.update(
    UV_PROJECT_ENVIRONMENT=str(ENV_DIR),
    UV_CACHE_DIR=str(TMP / "uv-cache"),
    HF_HOME=str(TMP / "hf"),  # the 9 GB of weights stay out of the saved output
    CUDA_VISIBLE_DEVICES="0",
    TOKENIZERS_PARALLELISM="false",
    PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
)
PY = str(ENV_DIR / "bin" / "python")
UV = [sys.executable, "-m", "uv"]

if not Path(PY).exists():
    if run([sys.executable, "-m", "pip", "install", "-q", "uv"]) != 0:
        raise RuntimeError("pip could not install uv")
    if (
        run(
            [*UV, "sync", "--frozen", "--no-dev", "--group", "ml", "--python", "3.12"],
            cwd=SRC,
            env=ENV,
        )
        != 0
    ):
        raise RuntimeError("uv sync failed: see the output above")

GPU_CHECK = (
    "import torch, sys\n"
    "print('torch', torch.__version__, 'CUDA', torch.version.cuda)\n"
    "x = torch.ones(64, 64, device='cuda', dtype=torch.float16)\n"
    "sys.exit(0 if float((x @ x)[0, 0]) == 64.0 else 1)\n"
)
if run([PY, "-c", GPU_CHECK], env=ENV) != 0:
    # The locked torch is built for CUDA 13 and needs a newer driver than this machine has.
    print(
        "The locked torch cannot use this GPU; installing the CUDA 12.6 build of the same version."
    )
    swap = [
        *UV,
        "pip",
        "install",
        "--python",
        PY,
        "--reinstall",
        "torch==2.14.0",
        "torchvision==0.29.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
        "--extra-index-url",
        "https://pypi.org/simple",
    ]
    if run(swap, env=ENV) != 0 or run([PY, "-c", GPU_CHECK], env=ENV) != 0:
        raise RuntimeError(
            "torch cannot use this GPU even with the CUDA 12.6 build; see the output above"
        )

In [ ]:
# ---- Fast kernels for the linear-attention layers: try, and fall back to the slow path -------
# flash-linear-attention is pure Python on triton. causal-conv1d compiles CUDA code, so it is only
# tried when nvcc exists. A tiny random Qwen3.5 runs a forward and backward pass in fp16 with
# whatever got installed; if that fails, the fast kernels are removed again.
KERNEL_CHECK = f"""
import sys
sys.path.insert(0, {str(SRC / "tests")!r})
import torch
from sft_fixture import tiny_qwen3_5_model
import importlib.util
model = tiny_qwen3_5_model().cuda().half()
ids = torch.randint(0, 1000, (1, 256), device="cuda")
loss = model(input_ids=ids, labels=ids).loss
loss.backward()
print("fla installed:", importlib.util.find_spec("fla") is not None, "causal_conv1d installed:", importlib.util.find_spec("causal_conv1d") is not None, "loss:", float(loss))
sys.exit(0 if torch.isfinite(loss) else 1)
"""


def kernel_check():
    return run([PY, "-c", KERNEL_CHECK], env=ENV) == 0


FAST_PACKAGES = ["flash-linear-attention", "fla-core", "einops"]
installed = run([*UV, "pip", "install", "--python", PY, "--no-deps", *FAST_PACKAGES], env=ENV) == 0
if installed and BUILD_CAUSAL_CONV1D and shutil.which("nvcc"):
    run([*UV, "pip", "install", "--python", PY, "--no-build-isolation", "causal-conv1d"], env=ENV)
if installed and kernel_check():
    FAST = True
    print("NOTE: fast linear-attention kernels are active.")
else:
    FAST = False
    run([*UV, "pip", "uninstall", "--python", PY, *FAST_PACKAGES, "causal-conv1d"], env=ENV)
    if not kernel_check():
        raise RuntimeError(
            "a tiny Qwen3.5 does not train in fp16 on this GPU even on the slow path"
        )
    print(
        "NOTE: the fast linear-attention kernels (flash-linear-attention, causal-conv1d) did not install or "
        "run on this GPU, so training continues on the slow torch path. It works, but is slower: the smoke "
        "run below measures how much."
    )

In [ ]:
# ---- Run ------------------------------------------------------------------------------------
if MODE not in ("smoke", "train"):
    raise ValueError("MODE must be 'smoke' or 'train'")
if MODE == "train" and PLATFORM == "kaggle":
    # The output of an earlier session, added to this notebook as an input, is read-only: copy it.
    staged = sft_platform.stage_resume(INPUT_ROOTS, RUN_NAME, RUN_DIR)
    if staged:
        print("continuing the run found in the inputs:", staged)

cmd = [PY, "scripts/sft_train.py"]
if PLAN.qlora:
    cmd.append("--qlora")
sets = [
    f"data.train_file={json.dumps(str(TRAIN_FILE))}",
    f"train.output_dir={json.dumps(str(RUN_DIR))}",
]
if MODE == "smoke":
    shutil.rmtree(OUT / f"{RUN_NAME}-smoke", ignore_errors=True)
    cmd.append("--smoke")
    sets.append(f"smoke.steps={SMOKE_STEPS}")
else:
    hours_left = SESSION_HOURS - (time.time() - T0) / 3600 - SAVE_MARGIN_HOURS
    if hours_left <= 0:
        raise RuntimeError("no time left in this session; start a new one and run again to resume")
    sets += ["train.resumable=true", f"train.time_limit_hours={hours_left:.2f}"]
    if (RUN_DIR / "run_config.json").exists():
        cmd.append("--resume")
        print("resuming the unfinished run in", RUN_DIR)
for item in [*sets, *EXTRA_SETS]:
    cmd += ["--set", item]

stamp = time.strftime("%Y%m%d-%H%M%S")
LOG = LOG_DIR / f"{RUN_NAME}-{MODE}-{stamp}.log"
code_ = run(cmd, log=LOG, cwd=SRC, env=ENV)
print(f"\nexit code {code_}; log: {LOG}")
if code_ != 0:
    raise RuntimeError("training failed: see the log above")

In [ ]:
# ---- Result ---------------------------------------------------------------------------------
if MODE == "smoke":
    smoke = json.loads((OUT / f"{RUN_NAME}-smoke" / "smoke.json").read_text())
    gpu = smoke["gpu"] or {}
    print(f"GPU:                  {gpu.get('device')} ({gpu.get('total_memory_gib')} GiB)")
    print(f"fast kernels:         {'yes' if FAST else 'no, slow torch path'}")
    print(f"precision:            {'QLoRA, 4-bit weights, fp16' if PLAN.qlora else 'bf16 LoRA'}")
    print(f"tokens per second:    {smoke['tokens_per_second']}")
    print(
        f"peak memory:          {gpu.get('peak_memory_allocated_gib')} GiB allocated, {gpu.get('peak_memory_reserved_gib')} GiB reserved"
    )
    print(f"losses:               {smoke['losses']}  finite: {smoke['loss_finite']}")
    print(
        f"steps with finite grad: {smoke['steps_with_finite_grad']} of {smoke['optimizer_steps']} (fp16 skips the first few)"
    )
    hours = smoke["projected_hours_per_epoch"]
    print(f"projected per epoch:  {hours} h over {smoke['epoch_tokens']:,} tokens")
    if not smoke["loss_finite"]:
        raise RuntimeError("the loss is not finite in this precision: do not start the full run")
    if not smoke["steps_with_finite_grad"]:
        raise RuntimeError(
            "every step overflowed: raise SMOKE_STEPS, or fp16 does not work for this model"
        )
    if hours:
        sessions = -(-hours // (SESSION_HOURS - SAVE_MARGIN_HOURS))
        print(f"one epoch takes {sessions:.0f} session(s) of {SESSION_HOURS} h")
        if PLATFORM == "kaggle":
            print(f"one epoch is {hours / 30:.0%} of Kaggle's 30 GPU hours a week")
else:
    summary_file = RUN_DIR / "train_summary.json"
    if summary_file.exists():
        summary = json.loads(summary_file.read_text())
        losses = [row["loss"] for row in summary["log_history"] if "loss" in row]
        print(
            f"done: adapter in {RUN_DIR / 'adapter'}, first loss {losses[0]:.3f}, last {losses[-1]:.3f}"
        )
    else:
        print("NOT FINISHED: the time limit stopped the run with a snapshot.")
        print(
            "Kaggle: save a version, then in a new session add this notebook's output as an input and run again."
        )
        print("Colab: reconnect and run again; the run continues from Drive.")
print("\nfiles in", OUT)
for path in sorted(OUT.glob("*")):
    print("  ", path.name)